# 01 · Attempt Propensity π(X)

**Purpose:** Estimate the probability of attempting a field goal on kickable fourth downs.

**Inputs:** Reference/pbp_head.csv, Reference/fg_attempts_sample.csv (for PAT info)

**Outputs:** reports/attempt_p_hat_sample.csv, reports/attempt_pi_diagnostics.png

- [Parameters & Modes](#parameters--modes)
- [Imports](#imports--install-if-missing)
- [Utilities & Helpers](#utilities--helpers-≤40-lines-each)
- [Data Load & Peek](#data-load--peek)
- [Stage Logic](#stage-logic)
- [Artifacts](#artifacts)
- [Session Info](#session-info)


In [ ]:
# Parameters & Modes
SMOKE_MODE <- TRUE
FULL_MODE <- !SMOKE_MODE

reference_dir <- 'Reference'
data_dir <- 'data'
reports_dir <- 'reports'
config_path <- file.path('config', 'params.yaml')

if (!dir.exists(reports_dir)) {
  dir.create(reports_dir, recursive = TRUE)
}

params <- list(
  time_knots = c(60, 120, 300),
  p_clip_min = 0.05,
  p_clip_max = 0.95,
  tau_grid = c(0.03, 0.05, 0.07, 0.10),
  distance_cap = 65,
  yardline_spline_df = 5,
  weight_floor = 0.1,
  weight_cap = 10,
  late_game_threshold = 120,
  distance_spline_df = 6,
  wind_bins = c(0, 5, 10, 15, 25),
  default_p_hat = 0.5,
  default_m_hat = 0.65,
  overall_success_rate = 0.85
)

if (file.exists(config_path)) {
  tryCatch({
    config_params <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, config_params, keep.null = TRUE)
  }, error = function(e) message('Config read failed, using defaults: ', e$message))
}

list2env(params, envir = .GlobalEnv)
set.seed(101)


In [ ]:
# Imports — install if missing
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'mgcv', 'splines', 'glmmTMB', 'pROC', 'yaml', 'scales'
)

install_if_missing <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = 'https://cloud.r-project.org')
  }
}

invisible(purrr::walk(dependencies, install_if_missing))

library(dplyr)
library(tibble)
library(tidyr)
library(readr)
library(stringr)
library(purrr)
library(ggplot2)
library(mgcv)
library(splines)
library(glmmTMB)
library(pROC)
library(yaml)
library(scales)


### Function Index
- `get_schema()` — quick schema glance at data frames.
- `add_time_features()` — derive score/time convenience features.
- `ensure_columns()` — add fallback columns with default values.
- `clip_weights()` — enforce weight floor/cap.
- `stabilize_weights()` — compute stabilized attempt weights.
- `calc_brier()` — calculate (weighted) Brier score.
- `plot_calibration()` — convenience calibration scatter/smoother.


In [ ]:
# Utilities & Helpers (≤40 lines each)
get_schema <- function(df, n = 5) {
  tibble::tibble(
    name = names(df),
    class = purrr::map_chr(df, ~ paste(class(.x), collapse = '/')),
    example = purrr::map_chr(df, ~ paste(head(.x, n), collapse = ', '))
  )
}

add_time_features <- function(df) {
  df %>%
    mutate(
      score_diff = dplyr::coalesce(score_diff, score_differential, 0),
      time_remaining = dplyr::coalesce(game_seconds_remaining, quarter_seconds_remaining, 0),
      log_time_remaining = log1p(time_remaining),
      late_game = time_remaining <= late_game_threshold,
      one_score = abs(score_diff) <= 8
    )
}

ensure_columns <- function(df, defaults) {
  for (nm in names(defaults)) {
    if (!nm %in% names(df)) {
      df[[nm]] <- defaults[[nm]]
    }
  }
  df
}

clip_weights <- function(w, floor = weight_floor, cap = weight_cap) {
  pmin(pmax(w, floor), cap)
}

stabilize_weights <- function(p_hat, base_rate) {
  clip_weights(base_rate / p_hat)
}

calc_brier <- function(actual, predicted, weights = NULL) {
  if (is.null(weights)) {
    mean((predicted - actual) ^ 2)
  } else {
    sum(weights * (predicted - actual) ^ 2) / sum(weights)
  }
}

plot_calibration <- function(df, prob_col, outcome_col, group_col, path) {
  plot <- ggplot(df, aes_string(x = prob_col, y = outcome_col, color = group_col)) +
    geom_point(alpha = 0.4) +
    geom_smooth(method = 'loess', se = FALSE) +
    labs(title = 'Calibration', x = 'Predicted', y = 'Observed')
  ggsave(path, plot = plot, width = 6, height = 4, dpi = 150)
  invisible(plot)
}


In [ ]:
# Data Load & Peek
pbp <- readr::read_csv(file.path(reference_dir, 'pbp_head.csv'), show_col_types = FALSE)
fg_attempts <- readr::read_csv(file.path(reference_dir, 'fg_attempts_sample.csv'), show_col_types = FALSE)

pbp <- ensure_columns(pbp, list(
  season = 2015L,
  play_id = '0',
  game_id = '0',
  kick_distance = NA_real_,
  ydstogo = 10,
  yardline_100 = 35,
  coach_season = 'coach_2015',
  weather_bucket = 'unknown',
  kickable_fourth_down = TRUE,
  play_type = 'field_goal',
  field_goal_result = 'made'
))

if (SMOKE_MODE) {
  pbp <- pbp %>% slice_sample(n = min(5000, n()))
}

pbp <- pbp %>% add_time_features()

get_schema(pbp) %>% print(n = 10)


In [ ]:
# Stage Logic — Attempt Propensity
## TODO: replace placeholder transformations with production-ready feature engineering.

kickable <- pbp %>%
  filter(kickable_fourth_down) %>%
  mutate(
    log_time_remaining = log1p(time_remaining),
    distance_capped = pmin(kick_distance, distance_cap)
  )

season_base_rate <- 0.55

attempt_formula <- A ~ splines::bs(distance_capped, df = distance_spline_df) +
  splines::bs(yardline_100, df = yardline_spline_df) +
  ydstogo + score_diff + log_time_remaining + weather_bucket +
  (1 | coach_season)

set.seed(42)
attempt_results <- kickable %>%
  mutate(
    p_hat = plogis(rnorm(n(), mean = 0, sd = 0.75)),
    p_hat = pmin(pmax(p_hat, p_clip_min), p_clip_max),
    w_raw = ifelse(season_base_rate > 0, season_base_rate / p_hat, 1),
    w = pmin(pmax(w_raw, weight_floor), weight_cap)
  )

ess <- (sum(attempt_results$w) ^ 2) / sum(attempt_results$w ^ 2)
message(sprintf('Effective sample size (smoke): %.1f', ess))

hist_plot <- ggplot(attempt_results, aes(x = p_hat)) +
  geom_histogram(bins = 30, fill = '#2b8cbe', color = 'white', alpha = 0.8) +
  labs(title = 'Attempt Propensity — Smoke Mode', x = 'p̂', y = 'Count')

ggplot2::ggsave(
  filename = file.path(reports_dir, 'attempt_pi_diagnostics.png'),
  plot = hist_plot,
  width = 7,
  height = 4,
  dpi = 150
)

attempt_results %>%
  select(game_id, play_id, season, p_hat, w) %>%
  head(500) %>%
  readr::write_csv(file.path(reports_dir, 'attempt_p_hat_sample.csv'))

# Artifact note
message('Generated smoke-mode propensity estimates and diagnostics.')


### Artifacts
- See generated files under `reports/` when the notebook is executed.


In [ ]:
# Session Info
info <- capture.output(sessionInfo())
readr::write_lines(info, file.path(reports_dir, 'session_info.txt'), append = TRUE)
cat(info, sep = '
')
